# Qwen Voice Studio — Kaggle + Cloudflare

A private-by-password text-to-speech studio powered by Qwen3-TTS from Hugging Face. The model runs on this Kaggle session's GPU. Cloudflare Quick Tunnel gives the interface a temporary internet URL while the session is running.

**Before running:** open Notebook settings and enable **Internet** and a **GPU** accelerator. The first run installs packages and downloads the model (several GB). Do not share the access password printed in the final cell. Stop the notebook session to close the tunnel.

In [ ]:
%pip -q install -U qwen-tts gradio soundfile
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /kaggle/working/cloudflared
!chmod +x /kaggle/working/cloudflared
print('Qwen3-TTS and Cloudflare Tunnel are ready.')

In [ ]:
import torch
from qwen_tts import Qwen3TTSModel

if not torch.cuda.is_available():
    raise RuntimeError('Enable a GPU in Kaggle Notebook settings, then rerun this cell.')

MODEL_ID = 'Qwen/Qwen3-TTS-12Hz-0.6B-CustomVoice'
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Downloading/loading {MODEL_ID} from Hugging Face…')
tts = Qwen3TTSModel.from_pretrained(
    MODEL_ID,
    device_map='cuda:0',
    dtype=torch.float16,
    attn_implementation='sdpa',
)
speakers = tts.get_supported_speakers()
languages = tts.get_supported_languages()
print('Model ready.')

In [ ]:
import secrets
import socket
import subprocess
import tempfile
import threading
import time
import uuid
from pathlib import Path

import gradio as gr
import soundfile as sf

MAX_CHARACTERS = 1800
with socket.socket() as port_probe:
    port_probe.bind(('127.0.0.1', 0))
    PORT = port_probe.getsockname()[1]

def generate_speech(text, speaker, language, direction):
    text = (text or '').strip()
    if not text:
        raise gr.Error('Write a line of text before generating speech.')
    if len(text) > MAX_CHARACTERS:
        raise gr.Error(f'Keep each generation under {MAX_CHARACTERS:,} characters.')
    with torch.inference_mode():
        wavs, sample_rate = tts.generate_custom_voice(
            text=text, language=language, speaker=speaker, instruct=(direction.strip() or None)
        )
    output = Path(tempfile.gettempdir()) / f'qwen-voice-{uuid.uuid4().hex}.wav'
    sf.write(output, wavs[0], sample_rate)
    return str(output), str(output)

css = '''
#studio {max-width: 1120px; margin: 0 auto;}
.hero {padding: 24px 4px 12px;}
.hero h1 {font-size: 2.55rem; letter-spacing: -0.05em; margin: 0 0 6px;}
.hero p {font-size: 1rem; opacity: .76;}
.generate button {min-height: 52px; font-size: 1rem; font-weight: 700;}
'''
theme = gr.themes.Soft(primary_hue='violet', secondary_hue='slate')
with gr.Blocks(title='Qwen Voice Studio') as demo:
    with gr.Column(elem_id='studio'):
        gr.HTML('<div class="hero"><h1>Qwen Voice Studio</h1><p>Write a line. Choose a voice. Hear it come alive.</p></div>')
        with gr.Row(equal_height=True):
            with gr.Column(scale=6):
                script = gr.Textbox(label='Your script', placeholder='Paste or write something to say …', lines=9, max_lines=18, max_length=MAX_CHARACTERS, value='Welcome to Qwen Voice Studio. Write a line, choose a voice, and bring your words to life.')
                count = gr.Markdown('**0** characters')
                script.input(lambda value: f'**{len(value or [])}** characters', inputs=script, outputs=count)
                with gr.Row():
                    voice = gr.Dropdown(speakers, value='Ryan' if 'Ryan' in speakers else speakers[0], label='Voice')
                    language = gr.Dropdown(languages, value='English' if 'English' in languages else languages[0], label='Language')
                direction = gr.Textbox(label='Delivery direction (optional)', placeholder='For example: Warm and reassuring, with a gentle pace', lines=2)
                create = gr.Button('Generate speech', variant='primary', elem_classes=['generate'])
            with gr.Column(scale=5):
                gr.Markdown('### Your audio\nThe first generation may take a little longer while the model warms up.')
                preview = gr.Audio(label='Preview', type='filepath', interactive=False)
                download = gr.File(label='Download WAV', interactive=False)
                gr.Markdown('Tip: A speaker’s native language usually gives the most natural result.')
        create.click(generate_speech, inputs=[script, voice, language, direction], outputs=[preview, download])

ACCESS_PASSWORD = secrets.token_urlsafe(12)
demo.launch(server_name='127.0.0.1', server_port=PORT, share=False, auth=('voice', ACCESS_PASSWORD), show_api=False, show_error=True, prevent_thread_lock=True, theme=theme, css=css)

tunnel = subprocess.Popen(
    ['/kaggle/working/cloudflared', 'tunnel', '--url', f'http://127.0.0.1:{PORT}'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)
def show_tunnel_logs():
    for line in tunnel.stdout:
        print(line.rstrip(), flush=True)
threading.Thread(target=show_tunnel_logs, daemon=True).start()
print('Studio login username: voice')
print('Studio access password (keep private):', ACCESS_PASSWORD)
print('The temporary https://…trycloudflare.com URL will appear in the tunnel logs above.')
print('Keep this cell running. Stop the Kaggle session to close the public URL.')


## Access and safety

Cloudflare Quick Tunnels are temporary demo URLs. The interface requires the username `voice` and a random password generated for this session. Keep both the URL and password private. Requests travel through Cloudflare Tunnel to the Kaggle runtime; speech generation runs in this notebook's GPU session. Stop the notebook session to end access.

Model: [Qwen/Qwen3-TTS-12Hz-0.6B-CustomVoice](https://huggingface.co/Qwen/Qwen3-TTS-12Hz-0.6B-CustomVoice) · [Qwen3-TTS usage guide](https://github.com/QwenLM/Qwen3-TTS) · [Cloudflare Quick Tunnels](https://developers.cloudflare.com/tunnel/get-started/quick-tunnels/)